# Interactive Web Map

**Author: Rohan Raval**

**Objectives:**
1. Create interactive Folium map
2. Add layer controls (deserts, services, HPI, transit)
3. Interactive popups with tract details
4. Export to HTML for sharing

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
from branca.colormap import LinearColormap
import json

pd.set_option('display.max_columns', 120)

candidates = [Path("."), Path(".."), Path("../..")]
for cand in candidates:
    if (cand / "data" / "processed").exists():
        PROJECT_ROOT = cand.resolve()
        break

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_SERVICES = DATA_DIR / "processed" / "services"
PROCESSED_ACS = DATA_DIR / "processed" / "acs5" / "2023"
TIGER_DIR = DATA_DIR / "external" / "tiger_tracts_2023"
RAW_SANDAG = DATA_DIR / "raw" / "sandag"
REPORTS = PROJECT_ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/rohanraval/Documents/GitHub/Mapping-Youth-Opportunity-Deserts


## Load Data

In [4]:
services = pd.read_csv(PROCESSED_SERVICES / "services_master_enhanced.csv")
metrics = pd.read_csv(PROCESSED_ACS / "tract_opportunity_desert_metrics_2023.csv", dtype={"GEOID": str})
transit = pd.read_csv(PROCESSED_SERVICES / "transit_stops_master.csv")

shp_path = TIGER_DIR / "tl_2023_06_tract.shp"
zip_path = TIGER_DIR / "tl_2023_06_tract.zip"
tracts_gdf = gpd.read_file(shp_path if shp_path.exists() else f"zip://{zip_path}")
sd_tracts = tracts_gdf[tracts_gdf["COUNTYFP"] == "073"].copy()
sd_tracts["GEOID"] = sd_tracts["GEOID"].astype(str)
sd_tracts = sd_tracts.merge(metrics, on="GEOID", how="left").to_crs("EPSG:4326")

hpi_zip = RAW_SANDAG / "Health_Places_Index_3_0_shapefile.zip"
if hpi_zip.exists():
    hpi_gdf = gpd.read_file(f"zip://{hpi_zip}")
    hpi_trim = hpi_gdf[["geoid", "hpi"]].rename(columns={"geoid": "GEOID", "hpi": "hpi_score"})
    hpi_trim["GEOID"] = hpi_trim["GEOID"].astype(str)
    sd_tracts = sd_tracts.merge(hpi_trim, on="GEOID", how="left")

print(f"Tracts: {len(sd_tracts)}")
print(f"Services: {len(services)}")
print(f"Transit stops: {len(transit)}")

Tracts: 912
Services: 520
Transit stops: 6173


## Create Base Map

In [5]:
center_lat = sd_tracts.geometry.centroid.y.mean()
center_lon = sd_tracts.geometry.centroid.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles='OpenStreetMap'
)

folium.TileLayer('CartoDB positron', name='Light Map').add_to(m)
folium.TileLayer('CartoDB dark_matter', name='Dark Map').add_to(m)

print(f"Base map centered at: ({center_lat:.4f}, {center_lon:.4f})")

Base map centered at: (32.8613, -117.1190)


/var/folders/t1/k9cqgpy95bs33y7tkwv_f69r0000gn/T/ipykernel_7280/372512065.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = sd_tracts.geometry.centroid.y.mean()
/var/folders/t1/k9cqgpy95bs33y7tkwv_f69r0000gn/T/ipykernel_7280/372512065.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = sd_tracts.geometry.centroid.x.mean()


## Add Tract Layers

In [6]:
desert_layer = folium.FeatureGroup(name='Opportunity Deserts', show=True)

deserts = sd_tracts[sd_tracts["desert_flag"] == 1].copy()

for idx, row in deserts.iterrows():
    popup_html = f"""
    <div style="font-family: Arial; font-size: 12px;">
        <b>{row['NAME']}</b><br>
        <hr style="margin: 5px 0;">
        <b>Youth (10-19):</b> {row['youth_10_19']:.0f} ({row['youth_10_19_per_1k']:.1f} per 1k)<br>
        <b>Services per 1k Youth:</b> {row['services_per_1k_youth_10_19']:.2f}<br>
        <b>Median Income:</b> ${row['median_hh_income_2023usd']:,.0f}<br>
        <b>Zero-Vehicle HH:</b> {100*row['zero_veh_share']:.1f}%<br>
        <b>Transit Stops (500m):</b> {row['stops_within_500m_per_tract']:.0f}
    </div>
    """
    
    folium.GeoJson(
        row['geometry'],
        style_function=lambda x: {
            'fillColor': '#d73027',
            'color': '#67000d',
            'weight': 2,
            'fillOpacity': 0.6
        },
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(desert_layer)

desert_layer.add_to(m)
print(f"Added {len(deserts)} desert tracts")

KeyError: 'NAME'

In [7]:
services_layer = folium.FeatureGroup(name='Service Access (Choropleth)', show=False)

colormap = LinearColormap(
    colors=['#fee5d9', '#fcae91', '#fb6a4a', '#de2d26', '#a50f15'],
    vmin=0,
    vmax=sd_tracts['services_per_1k_youth_10_19'].quantile(0.95),
    caption='Services per 1,000 Youth (ages 10-19)'
)

folium.GeoJson(
    sd_tracts,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties'].get('services_per_1k_youth_10_19', 0)),
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(fields=['NAME', 'services_per_1k_youth_10_19'],
                                   aliases=['Tract:', 'Services per 1k Youth:'])
).add_to(services_layer)

services_layer.add_to(m)
colormap.add_to(m)

In [8]:
if 'hpi_score' in sd_tracts.columns:
    hpi_layer = folium.FeatureGroup(name='Healthy Places Index', show=False)
    
    hpi_colormap = LinearColormap(
        colors=['#d73027', '#fc8d59', '#fee090', '#e0f3f8', '#91bfdb', '#4575b4'],
        vmin=sd_tracts['hpi_score'].min(),
        vmax=sd_tracts['hpi_score'].max(),
        caption='HPI Score (higher = better conditions)'
    )
    
    folium.GeoJson(
        sd_tracts.dropna(subset=['hpi_score']),
        style_function=lambda feature: {
            'fillColor': hpi_colormap(feature['properties']['hpi_score']),
            'color': 'grey',
            'weight': 0.3,
            'fillOpacity': 0.6
        },
        tooltip=folium.GeoJsonTooltip(fields=['NAME', 'hpi_score'],
                                       aliases=['Tract:', 'HPI Score:'])
    ).add_to(hpi_layer)
    
    hpi_layer.add_to(m)
    print("Added HPI layer")

Added HPI layer


## Add Service and Transit Points

In [9]:
service_colors = {
    'library': 'blue',
    'rec_center': 'green',
    'park': 'lightgreen',
    'ymca': 'purple'
}

service_icons = {
    'library': 'book',
    'rec_center': 'basketball-ball',
    'park': 'tree',
    'ymca': 'users'
}

service_groups = {}
for svc_type in service_colors.keys():
    service_groups[svc_type] = folium.FeatureGroup(
        name=f'{svc_type.replace("_", " ").title()}s',
        show=True
    )

services_with_coords = services.dropna(subset=['lat', 'lon'])

for idx, row in services_with_coords.iterrows():
    svc_type = row['service_type']
    if svc_type not in service_groups:
        continue
    
    popup_html = f"""
    <div style="font-family: Arial; font-size: 12px;">
        <b>{row['name']}</b><br>
        <i>{svc_type.replace('_', ' ').title()}</i><br>
        <hr style="margin: 5px 0;">
        <b>Org:</b> {row['org']}<br>
        <b>Address:</b> {row['address']}<br>
        <b>Youth-Serving:</b> {'Yes' if row['youth_serving'] else 'No'}
    </div>
    """
    
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color=service_colors[svc_type], icon=service_icons.get(svc_type, 'info-sign'), prefix='fa'),
        tooltip=row['name']
    ).add_to(service_groups[svc_type])

for group in service_groups.values():
    group.add_to(m)

print(f"Added {len(services_with_coords)} service markers")

Added 494 service markers


In [10]:
transit_layer = folium.FeatureGroup(name='Transit Stops', show=False)

transit_with_coords = transit.dropna(subset=['stop_lat', 'stop_lon']).sample(n=min(500, len(transit)))

for idx, row in transit_with_coords.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=3,
        color='orange',
        fill=True,
        fillOpacity=0.6,
        popup=f"<b>{row['stop_name']}</b><br>Agency: {row['stop_agency']}",
        tooltip=row['stop_name']
    ).add_to(transit_layer)

transit_layer.add_to(m)
print(f"Added {len(transit_with_coords)} transit stop markers (sample)")

Added 500 transit stop markers (sample)


## Add Map Controls and Plugins

In [11]:
folium.LayerControl(collapsed=False).add_to(m)

plugins.Fullscreen(
    position='topright',
    title='Fullscreen',
    title_cancel='Exit fullscreen'
).add_to(m)

plugins.MeasureControl(
    position='topleft',
    primary_length_unit='kilometers',
    primary_area_unit='sqkilometers'
).add_to(m)

plugins.LocateControl().add_to(m)

title_html = '''
<div style="position: fixed; 
            top: 10px; left: 60px; width: 400px; height: 90px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px;">
    <h4 style="margin-top:0;">Youth Opportunity Deserts - San Diego County</h4>
    <p style="margin:0;">Identifying areas with high youth population, low service access, and limited transportation.</p>
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

print("Added map controls and title")

Added map controls and title


## Save Interactive Map

In [12]:
output_path = REPORTS / "youth_opportunity_deserts_map.html"
m.save(str(output_path))

print(f"\nInteractive map saved to: {output_path}")
print(f"Open this file in a web browser to view the interactive map.")

TypeError: '>=' not supported between instances of 'float' and 'NoneType'